# Computer Vision per la "Smart Agriculture"

Questo notebook implementa una pipeline completa di **Transfer Learning** con **ResNet50** per la classificazione di immagini aeree da drone, nel contesto della startup AgTech *AgriFuture*.

L'obiettivo è riconoscere 5 tipologie di colture / stati del terreno partendo da immagini RGB di dimensione 224×224 px.

### Dataset

| Tipo | Dettaglio |
|------|-----------|
| **Sorgente** | Dati simulati (immagini random + label) — pipeline standalone |
| **Classi (5)** | Grano, Mais, Soia, Riso, Terreno vuoto |
| **Input shape** | `(224, 224, 3)` — standard ResNet50 |
| **Split** | 200 campioni training · 50 validazione |

### Pipeline

1. **Setup e Configurazione** — import, costanti, seed globale
2. **Generazione Dati Simulati** — funzione generatrice, distribuzione classi, visualizzazione campioni
3. **Architettura del Modello** — ResNet50 backbone + custom head, `model.summary()`
4. **Compilazione e Training** — Adam, `categorical_crossentropy`, callbacks, 3 epoche di prova
5. **Visualizzazione Risultati** — curve loss/accuracy train vs. validation
6. **Valutazione sul Validation Set** — classification report, confusion matrix, analisi errori
7. **(Bonus) Hyperparameter Tuning** — Keras Tuner: dropout rate e learning rate

## Setup e Configurazione

In [ ]:
# IMPORT E CONFIGURAZIONE GLOBALE

import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# --- Costanti ---
RANDOM_STATE  = 42
IMG_SIZE      = (224, 224)
IMG_SHAPE     = (224, 224, 3)
NUM_CLASSES   = 5
BATCH_SIZE    = 16          # ridotto rispetto al default — ResNet50 è più profonda di MobileNetV2
EPOCHS_TRIAL  = 5           # epoche per il training di prova (spec: 2-5)
DROPOUT_RATE  = 0.4         # tasso di dropout per la regolarizzazione
LEARNING_RATE = 1e-4        # lr basso: i pesi della head devono convergere senza destabilizzare il backbone
SAMPLES_TRAIN = 200         # campioni simulati training
SAMPLES_VAL   = 50          # campioni simulati validazione

CLASS_NAMES = ['Grano', 'Mais', 'Soia', 'Riso', 'Terreno vuoto']

# --- Seed globale per riproducibilità ---
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# --- Configurazione stile Matplotlib ---
plt.style.use('seaborn-v0_8-whitegrid')

mpl.rcParams.update({
    'figure.dpi'         : 120,
    'figure.facecolor'   : 'white',
    'font.family'        : 'sans-serif',
    'font.size'          : 11,
    'axes.titlesize'     : 13,
    'axes.titleweight'   : 'bold',
    'axes.labelsize'     : 11,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'lines.linewidth'    : 2,
    'lines.markersize'   : 7,
    'legend.frameon'     : True,
    'legend.framealpha'  : 0.9,
    'legend.fontsize'    : 10,
})

COLOR = {
    'blue'  : '#2196F3',
    'red'   : '#E91E63',
    'green' : '#4CAF50',
    'yellow': '#FF9800',
    'purple': '#9C27B0',
}

table_style = [
    {'selector': 'caption', 'props': [
        ('font-size', '13px'), ('font-weight', 'bold'),
        ('text-align', 'left'), ('padding-bottom', '6px')]},
    {'selector': 'th', 'props': [
        ('background-color', '#1565C0'), ('color', 'white'),
        ('font-size', '12px'), ('text-align', 'center'),
        ('padding', '6px 12px')]},
    {'selector': 'td', 'props': [
        ('text-align', 'center'), ('padding', '5px 12px'),
        ('font-size', '12px')]},
]

print('='*80)
print('SETUP COMPLETATO'.center(80))
print('='*80)
print(f'  TensorFlow  : {tf.__version__}')
print(f'  NumPy       : {np.__version__}')
print(f'  GPU         : {tf.config.list_physical_devices("GPU")}')
print(f'  Seed        : {RANDOM_STATE}')
print(f'  Classi      : {CLASS_NAMES}')
print('='*80)

## 1. Generazione Dati Simulati

### Funzione Generatrice e Struttura del Dataset

Lo script è **standalone**: nessun file esterno viene letto dal disco. La funzione `generate_dummy_data` produce immagini RGB casuali normalizzate in `[0, 1]` e le corrispondenti label in formato **one-hot**.

In un sistema reale le immagini verrebbero caricate con `tf.keras.utils.image_dataset_from_directory` o tramite un `tf.data.Dataset` alimentato da file su disco, con augmentation applicata *solo* sul training set (no leakage).

In [ ]:
# GENERAZIONE DATI SIMULATI

def generate_dummy_data(n_samples, img_size, num_classes, seed=RANDOM_STATE):
    """
    Genera un dataset simulato di immagini e label one-hot encoded.

    Parametri
    ---------
    n_samples   : int   — numero di campioni da generare
    img_size    : tuple — (h, w) dimensioni spaziali dell'immagine
    num_classes : int   — numero di classi target
    seed        : int   — seed per la riproducibilità

    Ritorna
    -------
    X : np.ndarray shape (n_samples, h, w, 3) — immagini float32 in [0, 1]
    y : np.ndarray shape (n_samples, num_classes) — label one-hot float32
    """
    rng = np.random.default_rng(seed)
    h, w = img_size
    # Immagini random normalizzate [0,1] — simulano pixel RGB da drone
    X = rng.random((n_samples, h, w, 3)).astype(np.float32)
    # Label intere uniformemente distribuite tra le classi
    labels = rng.integers(0, num_classes, size=n_samples)
    # One-hot encoding tramite Keras
    y = tf.keras.utils.to_categorical(labels, num_classes=num_classes).astype(np.float32)
    return X, y


# --- Creazione dataset ---
print('='*80)
print('GENERAZIONE DATI SIMULATI'.center(80))
print('='*80 + '\n')

X_train, y_train = generate_dummy_data(SAMPLES_TRAIN, IMG_SIZE, NUM_CLASSES, seed=RANDOM_STATE)
X_val,   y_val   = generate_dummy_data(SAMPLES_VAL,   IMG_SIZE, NUM_CLASSES, seed=RANDOM_STATE + 1)

print(f'  Training set   : X={X_train.shape}  y={y_train.shape}')
print(f'  Validation set : X={X_val.shape}   y={y_val.shape}')
print(f'  Dtype          : {X_train.dtype}')
print(f'  Range pixel    : [{X_train.min():.3f}, {X_train.max():.3f}]')
print('-'*80 + '\n')

# Distribuzione classi
labels_train = y_train.argmax(axis=1)
labels_val   = y_val.argmax(axis=1)

print('  Distribuzione classi — Training:')
for i, name in enumerate(CLASS_NAMES):
    n = (labels_train == i).sum()
    print(f'    {name:<15}: {n:>3} campioni ({n/SAMPLES_TRAIN:.1%})')
print()
print('  Distribuzione classi — Validazione:')
for i, name in enumerate(CLASS_NAMES):
    n = (labels_val == i).sum()
    print(f'    {name:<15}: {n:>3} campioni ({n/SAMPLES_VAL:.1%})')
print('='*80)

In [ ]:
# VISUALIZZAZIONE CAMPIONI SIMULATI

fig, axes = plt.subplots(1, 5, figsize=(16, 4.5))
fig.suptitle(
    'Campioni Simulati — Dataset Smart Agriculture\n(immagini random: pixel uniformi [0,1] per classe)',
    fontsize=16, fontweight='bold'
)

for i, ax in enumerate(axes.flatten()):
    img       = X_train[i]
    label_idx = y_train[i].argmax()
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[label_idx], fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

### Osservazioni

I 200 campioni di training e 50 di validazione sono **immagini random** con pixel estratti da una distribuzione uniforme in `[0, 1]`: non contengono alcuna struttura visiva reale. Le label sono assegnate casualmente con distribuzione approssimativamente uniforme tra le 5 classi (~20% per classe), simulando una raccolta bilanciata tipica di un dataset curato.

Questa scelta è funzionale allo scopo dell'esercizio (testare l'architettura end-to-end senza dipendenze esterne), ma implica che il modello non riuscirà a imparare pattern visivi significativi: le metriche di accuracy sul validation set rimarranno vicine al **20% (baseline casuale per 5 classi)**. In un deploy reale le immagini aeree conterrebbero texture distintive per ogni coltura.

## 2. Architettura del Modello

### Costruzione con ResNet50 (Transfer Learning — Feature Extraction)

La strategia adottata è la **Feature Extraction**: il backbone ResNet50 pre-addestrato su ImageNet viene usato come estrattore di feature congelato (`trainable=False`). Solo la *custom head* aggiunta sopra viene addestrata da zero.

```
Input (224, 224, 3)
    │
ResNet50 (frozen) ← pesi ImageNet
    │
GlobalAveragePooling2D  ← riduce l'output spaziale a un vettore 1D
    │
Dropout(0.4)            ← regolarizzazione per ridurre overfitting
    │
Dense(5, softmax)       ← classificatore finale per 5 classi
```

In [ ]:
# COSTRUZIONE DEL MODELLO — RESNET50 + CUSTOM HEAD

def build_model(img_shape, num_classes, dropout_rate):
    """
    Costruisce il modello di classificazione basato su ResNet50.

    Architettura:
        - Backbone: ResNet50 pre-addestrata su ImageNet (include_top=False, trainable=False)
        - Head:     GlobalAveragePooling2D → Dropout → Dense(softmax)

    Parametri
    ---------
    img_shape    : tuple — forma dell'input (h, w, canali), es. (224, 224, 3)
    num_classes  : int   — numero di classi del classificatore finale
    dropout_rate : float — tasso di dropout [0, 1)

    Ritorna
    -------
    model : tf.keras.Model — modello compilato non ancora addestrato
    """
    # --- Backbone ResNet50 ---
    backbone = tf.keras.applications.ResNet50(
        include_top=False,        # rimuove il classificatore ImageNet originale
        weights='imagenet',       # pesi pre-addestrati su ImageNet
        input_shape=img_shape,
    )
    # Congelamento dei pesi: preserva la conoscenza acquisita su ImageNet
    backbone.trainable = False

    # --- Custom Head ---
    inputs = tf.keras.Input(shape=img_shape)
    x = backbone(inputs, training=False)  # training=False: BN layers in inference mode
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='SmartAgriculture_ResNet50')
    return model

In [ ]:
# ISTANZIAZIONE E SUMMARY DEL MODELLO

model = build_model(IMG_SHAPE, NUM_CLASSES, DROPOUT_RATE)

print('='*80)
print('ARCHITETTURA DEL MODELLO'.center(80))
print('='*80 + '\n')

model.summary()

print('\n' + '-'*80)
total_params     = model.count_params()
trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
frozen_params    = total_params - trainable_params

print(f'\n  Parametri totali       : {total_params:>12,}')
print(f'  Parametri addestrabili : {trainable_params:>12,}  ({trainable_params/total_params:.1%})')
print(f'  Parametri congelati    : {frozen_params:>12,}  ({frozen_params/total_params:.1%})')
print('\n' + '='*80)

### Osservazioni — Parametri Totali vs Addestrabili

Il `model.summary()` mostra la differenza fondamentale del Transfer Learning:

| Categoria | Parametri | % sul totale |
|---|---|---|
| **Totali** | ~25.6M | 100% |
| **Congelati** (ResNet50) | ~25.6M | ~99.9% |
| **Addestrabili** (head) | ~10.2K | ~0.04% |

Solo i ~10.000 parametri della custom head vengono aggiornati durante il training. Questo rende l'addestramento **estremamente rapido e stabile**: non si rischia di distruggere la rappresentazione visiva appressa su ImageNet (*catastrophic forgetting*).

### Motivazione delle Scelte Architetturali

| Scelta | Motivazione |
|---|---|
| **ResNet50** | Backbone solido con connessioni residuali che evitano il vanishing gradient; standard per task di CV su immagini 224×224 |
| **`include_top=False`** | Rimuove il classificatore ImageNet (1000 classi) per sostituirlo con la head personalizzata a 5 classi |
| **`trainable=False`** | Feature extraction pura: i pesi ImageNet vengono preservati, riducendo drasticamente il rischio di overfitting su dati limitati |
| **`training=False` nel forward pass** | I layer Batch Normalization di ResNet50 restano in *inference mode* anche durante il training, evitando aggiornamenti alle statistiche BN congelate |
| **GlobalAveragePooling2D** | Preferito a `Flatten` perché riduce l'output spaziale `(7, 7, 2048)` a un vettore `(2048,)` mediando spazialmente, abbattendo i parametri della head e agendo come regolarizzatore implicito |
| **Dropout(0.4)** | Tasso intermedio: sufficiente a contrastare l'overfitting su un dataset piccolo (200 campioni) senza penalizzare eccessivamente la capacità di apprendimento |
| **Dense(5, softmax)** | Output a 5 neuroni con attivazione softmax per produrre distribuzioni di probabilità normalizzate tra le 5 classi |
| **Learning rate = 1e-4** | Valore basso per la fase di feature extraction: la head è nuova e potrebbe generare gradienti instabili se lr troppo alto |

## 3. Compilazione e Training

In [ ]:
# FUNZIONE DI TRAINING

def train_model(model, X_train, y_train, X_val, y_val,
                learning_rate=LEARNING_RATE,
                batch_size=BATCH_SIZE,
                epochs=EPOCHS_TRIAL):
    """
    Compila e addestra il modello con Adam, categorical_crossentropy e callbacks standard.

    Parametri
    ---------
    model         : tf.keras.Model — modello da addestrare
    X_train       : np.ndarray     — immagini di training
    y_train       : np.ndarray     — label one-hot di training
    X_val         : np.ndarray     — immagini di validazione
    y_val         : np.ndarray     — label one-hot di validazione
    learning_rate : float          — learning rate per Adam
    batch_size    : int            — dimensione del batch
    epochs        : int            — numero massimo di epoche

    Ritorna
    -------
    history : tf.keras.callbacks.History — storico del training
    """
    # Compilazione
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    # Callbacks standard (senza ModelCheckpoint: nessun file salvato su disco)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=10, restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5,
            min_lr=1e-7, verbose=1,
        ),
    ]

    print('='*80)
    print('TRAINING'.center(80))
    print('='*80)
    print(f'  Ottimizzatore : Adam (lr={learning_rate})')
    print(f'  Loss          : categorical_crossentropy')
    print(f'  Batch size    : {batch_size}')
    print(f'  Epoche        : {epochs}')
    print('='*80 + '\n')

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1,
    )
    return history

In [ ]:
# AVVIO TRAINING DI PROVA

history = train_model(model, X_train, y_train, X_val, y_val)

### Osservazioni — Andamento del Training

Con sole 5 epoche su dati completamente casuali, i valori di loss e accuracy non mostrano una convergenza significativa: la **val_accuracy** rimarrà intorno al 20% (equivalente a una scelta casuale tra 5 classi).

Questo è il comportamento atteso: ResNet50 estrae feature da immagini reali con struttura visiva; su pixel random non esiste alcun segnale discriminativo che la head possa sfruttare. In un contesto reale (immagini aeree vere), la val_accuracy sulle prime epoche sarebbe già superiore all'80% grazie alla qualità dei feature estratti da ImageNet.

L'**EarlyStopping** (patience=10) non scatterà con sole 3 epoche di prova; il **ReduceLROnPlateau** (patience=5) potrebbe abbassare il lr se la val_loss non migliora.

## 4. Visualizzazione Risultati

### Curve di Training vs. Validation (Loss e Accuracy)

In [ ]:
# CURVE DI TRAINING vs. VALIDATION

def plot_history(history, title_suffix=''):
    """
    Plotta le curve di loss e accuracy su training e validation set.

    Parametri
    ---------
    history      : tf.keras.callbacks.History — storico restituito da model.fit()
    title_suffix : str — suffisso opzionale per il titolo (es. nome del modello)
    """
    epochs_range = range(1, len(history.history['loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    suffix = f' — {title_suffix}' if title_suffix else ''
    fig.suptitle(f'Curve di Training vs. Validation{suffix}', fontsize=14, fontweight='bold')

    # --- Loss ---
    ax1 = axes[0]
    ax1.plot(epochs_range, history.history['loss'],
             color=COLOR['blue'], marker='o', markersize=5, label='Training Loss')
    ax1.plot(epochs_range, history.history['val_loss'],
             color=COLOR['red'], marker='s', markersize=5, linestyle='--', label='Validation Loss')
    ax1.set_title('Loss')
    ax1.set_xlabel('Epoca')
    ax1.set_ylabel('Categorical Crossentropy')
    ax1.set_xticks(epochs_range)
    ax1.legend()

    # --- Accuracy ---
    ax2 = axes[1]
    ax2.plot(epochs_range, history.history['accuracy'],
             color=COLOR['blue'], marker='o', markersize=5, label='Training Accuracy')
    ax2.plot(epochs_range, history.history['val_accuracy'],
             color=COLOR['red'], marker='s', markersize=5, linestyle='--', label='Validation Accuracy')
    # Linea baseline casuale (1/NUM_CLASSES)
    ax2.axhline(1 / NUM_CLASSES, color='gray', linewidth=1.2, linestyle=':',
                label=f'Baseline casuale (1/{NUM_CLASSES} = {1/NUM_CLASSES:.0%})')
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Epoca')
    ax2.set_ylabel('Accuracy')
    ax2.set_xticks(epochs_range)
    ax2.set_ylim(0, 1)
    ax2.legend()

    plt.tight_layout()
    plt.show()


plot_history(history, title_suffix='Feature Extraction ResNet50')

### Osservazioni — Diagnosi Overfitting / Underfitting

Le curve permettono di diagnosticare tre scenari tipici:

| Scenario | Segnale nelle curve |
|---|---|
| **Underfitting** | Training loss alta e stagnante; gap minimo tra train e val |
| **Overfitting** | Training loss scende, val_loss risale (divergenza) |
| **Buona generalizzazione** | Entrambe le curve scendono e convergono |

Con dati simulati casualmente su 5 epoche ci si aspetta il caso **underfitting**: sia training che validation accuracy oscillano intorno alla baseline del 20% (linea tratteggiata grigia), senza segnali di apprendimento. Questo è il comportamento atteso e corretto — conferma che l'architettura è implementata bene ma il dataset non offre segnale discriminativo.

In produzione, con immagini aeree reali, si osserverebbe una rapida discesa della loss nelle prime epoche (il backbone già conosce bordi, texture e strutture) seguita da convergenza stabile.

## 5. Valutazione sul Validation Set

### Metriche Aggregate e Analisi degli Errori

In [ ]:
# VALUTAZIONE SUL VALIDATION SET

def evaluate_model(model, X_val, y_val, class_names):
    """
    Valuta il modello sul validation set: accuracy, classification report e confusion matrix.

    Parametri
    ---------
    model       : tf.keras.Model — modello addestrato
    X_val       : np.ndarray     — immagini di validazione
    y_val       : np.ndarray     — label one-hot di validazione
    class_names : list[str]      — nomi delle classi in ordine

    Ritorna
    -------
    y_pred : np.ndarray — predizioni come indici di classe (int)
    y_true : np.ndarray — label vere come indici di classe (int)
    """
    print('='*80)
    print('VALUTAZIONE SUL VALIDATION SET'.center(80))
    print('='*80 + '\n')

    # Predizioni probabilistiche → classe con probabilità massima
    y_prob = model.predict(X_val, verbose=0)
    y_pred = y_prob.argmax(axis=1)
    y_true = y_val.argmax(axis=1)

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f'  Val Loss     : {val_loss:.4f}')
    print(f'  Val Accuracy : {val_acc:.4f}  ({val_acc:.1%})')
    print(f'  Baseline     : {1/len(class_names):.4f}  ({1/len(class_names):.1%})')
    print('-'*80 + '\n')

    # Classification Report
    print('Classification Report:\n')
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    return y_pred, y_true


y_pred, y_true = evaluate_model(model, X_val, y_val, CLASS_NAMES)

In [ ]:
# CONFUSION MATRIX E VISUALIZZAZIONE PREDIZIONI

# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
fig.suptitle('Valutazione sul Validation Set', fontsize=14, fontweight='bold')

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, square=True, ax=ax,
    annot_kws={'size': 11}
)
ax.set_title('Confusion Matrix — Validation Set')
ax.set_xlabel('Classe predetta')
ax.set_ylabel('Classe reale')
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)

plt.show()


### Osservazioni — Analisi degli Errori

La confusion matrix mostra la distribuzione delle predizioni su ciascuna coppia (classe reale, classe predetta). Con dati random, ci si aspetta una distribuzione approssimativamente uniforme su tutte le 50 righe della matrice — senza pattern diagonali evidenti che indicherebbero apprendimento.

**Analisi qualitativa degli errori:**
- Gli errori non seguono alcun pattern sistematico (es. "Mais confuso con Grano"): sono distribuiti casualmente, confermando l'assenza di feature visive significative.
- In un dataset reale, la confusion matrix evidenzierebbe le coppie di classi visivamente simili (es. Riso vs Grano per la struttura fogliare), su cui concentrare l'augmentation o aggiustamenti della soglia.
- Le predizioni con **bordo rosso** nella griglia mostrano campioni classificati erroneamente: l'analisi visiva conferma che non esiste alcuna caratteristica visiva riconoscibile che differenzi le classi nelle immagini simulate.

## 6. (Bonus) Hyperparameter Tuning con Keras Tuner

### Strategia di Tuning

L'obiettivo è trovare la combinazione ottimale di due iperparametri critici della **custom head**:

| Iperparametro | Range esplorato | Motivazione |
|---|---|---|
| `dropout_rate` | 0.2, 0.3, 0.4, 0.5 | Controlla la regolarizzazione: valori bassi = più capacità, valori alti = meno overfitting |
| `learning_rate` | 1e-3, 1e-4, 1e-5 | Velocità di convergenza della head; lr troppo alto può destabilizzare l'ottimizzazione |

**Strategia**: `RandomSearch` con 6 trial e 2 epoche ciascuno — sufficiente per confrontare tendenze su dati simulati senza costo computazionale elevato.

> **Installazione**: `pip install keras-tuner`

In [ ]:
# FUNZIONE BUILD PER KERAS TUNER

import tempfile
import keras_tuner as kt

# Costanti per la ricerca
TUNER_MAX_TRIALS  = 6    # numero di combinazioni da esplorare
TUNER_EPOCHS      = 2    # epoche per trial (ridotte per velocità su dati simulati)
TUNER_PROJECT     = 'smart_agriculture'


def build_model_tuner(hp):
    """
    Funzione di build compatibile con Keras Tuner: definisce lo spazio degli iperparametri.

    Parametri
    ---------
    hp : kt.HyperParameters — oggetto Keras Tuner per definire l'hp space

    Ritorna
    -------
    model : tf.keras.Model — modello compilato con la combinazione hp corrente
    """
    # --- Spazio degli iperparametri ---
    dropout_rate  = hp.Choice('dropout_rate',  values=[0.2, 0.3, 0.4, 0.5])
    learning_rate = hp.Choice('learning_rate', values=[1e-3, 1e-4, 1e-5])

    # --- Backbone ResNet50 congelato ---
    backbone = tf.keras.applications.ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=IMG_SHAPE,
    )
    backbone.trainable = False

    # --- Custom Head con hp correnti ---
    inputs  = tf.keras.Input(shape=IMG_SHAPE)
    x       = backbone(inputs, training=False)
    x       = tf.keras.layers.GlobalAveragePooling2D()(x)
    x       = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

In [ ]:
# AVVIO RICERCA HYPERPARAMETER TUNING

# Directory temporanea: rimossa automaticamente alla fine della sessione (nessun file residuo)
_tuner_dir = tempfile.mkdtemp()

tuner = kt.RandomSearch(
    hypermodel=build_model_tuner,
    objective='val_loss',
    max_trials=TUNER_MAX_TRIALS,
    seed=RANDOM_STATE,
    directory=_tuner_dir,
    project_name=TUNER_PROJECT,
    overwrite=True,
)

print('='*80)
print('HYPERPARAMETER TUNING — RandomSearch'.center(80))
print('='*80)
print(f'  Spazio: dropout_rate ∈ [0.2, 0.3, 0.4, 0.5]')
print(f'          learning_rate ∈ [1e-3, 1e-4, 1e-5]')
print(f'  Trial  : {TUNER_MAX_TRIALS}')
print(f'  Epoche : {TUNER_EPOCHS} per trial')
print('='*80 + '\n')

tuner.search(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=TUNER_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)

print('\n' + '='*80)
print('RICERCA COMPLETATA'.center(80))
print('='*80)

In [ ]:
# RISULTATI TUNING — BEST MODEL vs BASELINE

# --- Recupero best hyperparameters ---
best_hp    = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.get_best_models(num_models=1)[0]   # get_best_models usa num_models, non num_trials

print('='*80)
print('RISULTATI HYPERPARAMETER TUNING'.center(80))
print('='*80 + '\n')

print('  Best Hyperparameters:')
print(f'    dropout_rate  : {best_hp.get("dropout_rate")}')
print(f'    learning_rate : {best_hp.get("learning_rate")}')
print('-'*80 + '\n')

# --- Confronto baseline vs best model ---
baseline_loss, baseline_acc = model.evaluate(X_val, y_val, verbose=0)
best_loss,     best_acc     = best_model.evaluate(X_val, y_val, verbose=0)

print('  Confronto sul Validation Set:')
print(f'  {"Modello":<30} {"Val Loss":>10} {"Val Accuracy":>14}')
print(f'  {"-"*54}')
print(f'  {"Baseline (dropout=0.4, lr=1e-4)":<30} {baseline_loss:>10.4f} {baseline_acc:>14.4f}')
print(f'  {"Best Model (tuned)":<30} {best_loss:>10.4f} {best_acc:>14.4f}')
print('='*80 + '\n')

# --- Riepilogo tutti i trial ---
print('  Riepilogo tutti i trial:\n')
results = []
for trial in tuner.oracle.trials.values():
    hp_vals  = trial.hyperparameters.values
    score    = trial.score
    results.append({
        'Trial'        : trial.trial_id,
        'dropout_rate' : hp_vals.get('dropout_rate', '-'),
        'learning_rate': hp_vals.get('learning_rate', '-'),
        'Val Loss'     : round(score, 4) if score is not None else None,
    })

df_results = (
    pd.DataFrame(results)
    .sort_values('Val Loss')
    .reset_index(drop=True)
)

display(
    df_results.style
        .format({'Val Loss': '{:.4f}', 'learning_rate': '{:.0e}'})
        .set_caption(f'Keras Tuner — RandomSearch ({TUNER_MAX_TRIALS} trial)')
        .set_table_styles(table_style)
        .hide(axis='index')
)

# --- Addestramento best model completo (EPOCHS_TRIAL) e curve ---
print('\nAddestramento best model per confronto curve...')
history_best = best_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_TRIAL,
    verbose=0,
)

plot_history(history_best, title_suffix=f'Best Model (dropout={best_hp.get("dropout_rate")}, lr={best_hp.get("learning_rate")})')

### Osservazioni — Impatto degli Iperparametri

La tabella dei trial mostra il risultato della ricerca su 6 combinazioni casuali di `dropout_rate` e `learning_rate`. Su dati simulati casualmente le differenze tra trial saranno trascurabili (varianza del rumore domina), ma la struttura dell'esperimento rimane metodologicamente corretta.

**Interpretazione attesa su dati reali:**

- **`learning_rate`** è tipicamente l'iperparametro con maggiore impatto: un lr troppo alto (1e-3) può portare a oscillazioni della loss, mentre lr troppo basso (1e-5) rallenta inutilmente la convergenza. Il valore ottimale per feature extraction con testa nuova è solitamente **1e-4**.
- **`dropout_rate`** agisce come regolarizzatore implicito: con dataset piccoli (< 1000 campioni) un valore più alto (0.4–0.5) riduce l'overfitting; con dataset ampi si può abbassare senza rischi.

**Passo successivo — Fine Tuning**: una volta identificata la migliore configurazione della head con feature extraction, si può sbloccare progressivamente gli ultimi blocchi di ResNet50 (`backbone.trainable = True` + un lr molto basso, es. 1e-5) per adattare i pesi del backbone al dominio specifico delle immagini aeree agricole.